# Week 1: SmolVLA Fine-tuning Pipeline Setup

**Goal:** End-to-end pipeline working — model loads, dataset loads, one forward pass returns valid actions. No fine-tuning yet.

**Stack:** SmolVLA (450M VLA) + LeRobot-native dataset (`lerobot/svla_so100_pickplace`). No external simulators.

**Plan for the project:**
- Week 1 (today): Pipeline works end-to-end.
- Week 2: Run inference on test split, measure baseline action-prediction error.
- Week 3-4: LoRA fine-tune. Multiple ranks (8/16/32). Compare.
- Week 5: Efficiency contribution (INT8 quantization OR action chunking). Latency vs accuracy Pareto curve.
- Week 6: Writeup, demo video, clean repo.

## Cell 1: Runtime check

In [ ]:
!nvidia-smi
import torch, sys
print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 2: VRAM monitor helper

Call `vram()` after major steps. Save the numbers for the writeup.

In [ ]:
def vram():
    if not torch.cuda.is_available():
        print("No GPU"); return
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {used:.2f}GB allocated, {reserved:.2f}GB reserved (of {total:.0f}GB)")

vram()

## Cell 3: Mount Google Drive

All checkpoints and outputs go to Drive so they survive Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/vla-project/{checkpoints,outputs,logs}
!ls /content/drive/MyDrive/vla-project/

## Cell 4: Install lerobot with SmolVLA + PEFT support

Takes ~2-3 minutes. Watch for `Successfully installed` at the end.

In [ ]:
!pip install -q "lerobot[smolvla,peft]"
!pip install -q imageio[ffmpeg]

print("\n--- Versions ---")
import lerobot
print(f"lerobot: {lerobot.__version__}")

## Cell 5: Hugging Face login (via Colab Secrets)

**One-time setup before running this cell:**
1. Get a token at https://huggingface.co/settings/tokens (Write access).
2. In Colab, click the 🔑 icon in the left sidebar.
3. Add a new secret named `HF_TOKEN`, paste your token, toggle "Notebook access" ON.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
print("Logged in to Hugging Face Hub")

## Cell 6: Load the SmolVLA model

First time: downloads ~1GB. Subsequent runs: instant (cached).

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

policy = SmolVLAPolicy.from_pretrained("lerobot/smolvla_base")
policy = policy.to("cuda")
policy.eval()

n_params = sum(p.numel() for p in policy.parameters())
n_trainable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
print(f"SmolVLA loaded: {n_params/1e6:.1f}M total params, {n_trainable/1e6:.1f}M trainable")
vram()

## Cell 7: Inspect policy configuration

Tells us what input features the policy expects — critical for matching with the dataset.

In [ ]:
print("=== Policy Config ===")
print(f"\nInput features:")
for k, v in policy.config.input_features.items():
    print(f"  {k}: shape={v.shape}, type={v.type}")

print(f"\nOutput features:")
for k, v in policy.config.output_features.items():
    print(f"  {k}: shape={v.shape}, type={v.type}")

print(f"\nAction chunk size: {policy.config.chunk_size}")
print(f"N action steps to predict: {policy.config.n_action_steps}")

## Cell 8: Load a SmolVLA-compatible dataset

We use `lerobot/svla_so100_pickplace` — the SO-100 robot arm pick-and-place dataset that SmolVLA was evaluated on. Already in LeRobot format. First time downloads ~few GB, cached after.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_REPO_ID = "lerobot/svla_so100_pickplace"

dataset = LeRobotDataset(DATASET_REPO_ID)
print(f"Dataset loaded: {DATASET_REPO_ID}")
print(f"Number of episodes: {dataset.num_episodes}")
print(f"Number of frames: {dataset.num_frames}")
print(f"FPS: {dataset.fps}")

print(f"\nDataset features:")
for k, v in dataset.features.items():
    shape = v.get('shape', '?')
    dtype = v.get('dtype', '?')
    print(f"  {k}: shape={shape}, dtype={dtype}")

## Cell 9: Inspect a single dataset sample

Confirm we can pull a frame and see what's inside before passing it to the policy.

In [ ]:
import numpy as np

sample = dataset[0]
print("Sample keys and tensor info:\n")
for k, v in sample.items():
    if hasattr(v, 'shape'):
        print(f"  {k}: shape={tuple(v.shape)}, dtype={v.dtype}")
    elif isinstance(v, (str, int, float)):
        print(f"  {k}: {v}")
    else:
        print(f"  {k}: {type(v).__name__}")

## Cell 10: View a sample image

Visual sanity check — the robot's camera view should look like a tabletop with objects.

In [ ]:
import matplotlib.pyplot as plt

# Find an image key (varies by dataset; common names: 'observation.image', 'observation.images.front', etc.)
image_keys = [k for k in sample.keys() if 'image' in k.lower()]
print(f"Found image keys: {image_keys}\n")

if image_keys:
    img_key = image_keys[0]
    img = sample[img_key]
    # LeRobot images are typically (C, H, W) float [0,1] tensors
    if img.dim() == 3 and img.shape[0] in (1, 3):
        img_to_show = img.permute(1, 2, 0).numpy()
    else:
        img_to_show = img.numpy()
    plt.figure(figsize=(6, 6))
    plt.imshow(img_to_show)
    plt.title(f"{img_key}\nshape={tuple(img.shape)}")
    plt.axis('off')
    plt.show()

    if 'task' in sample:
        print(f"Task instruction: {sample['task']}")

## Cell 11: Run one forward pass through the policy (THE MOMENT OF TRUTH)

This is the end-to-end pipeline test. If this returns an action tensor without errors, **Week 1 is done**.

In [ ]:
import torch

# Build a batch from a single sample (add batch dimension)
sample = dataset[0]

batch = {}
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        batch[k] = v.unsqueeze(0).to("cuda")
    elif isinstance(v, str):
        batch[k] = [v]  # tasks are usually lists of strings
    else:
        batch[k] = v

print("Batch keys:", list(batch.keys()))

# Predict an action
with torch.no_grad():
    try:
        action = policy.select_action(batch)
        print(f"\n✅ SUCCESS! Predicted action: shape={tuple(action.shape)}, dtype={action.dtype}")
        print(f"Action values (first 7): {action.cpu().numpy().flatten()[:7]}")
    except Exception as e:
        print(f"\n❌ Error: {type(e).__name__}: {e}")
        print("\nDebug — what the policy expects:")
        for k, v in policy.config.input_features.items():
            print(f"  {k}: shape={v.shape}")
        print("\nWhat the batch has:")
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                print(f"  {k}: shape={tuple(v.shape)}")

vram()

## Cell 12: Save Week 1 status report

Write a small markdown summary to Drive so you have a record of what worked.

In [ ]:
import json, datetime

report = {
    "date": datetime.datetime.now().isoformat(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0,
    "lerobot_version": lerobot.__version__,
    "model": "lerobot/smolvla_base",
    "params_total_M": round(sum(p.numel() for p in policy.parameters()) / 1e6, 1),
    "dataset": DATASET_REPO_ID,
    "dataset_episodes": dataset.num_episodes,
    "dataset_frames": dataset.num_frames,
    "week1_status": "pipeline_works",
}

out_path = "/content/drive/MyDrive/vla-project/logs/week1_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to: {out_path}")
print(json.dumps(report, indent=2))

## ✅ End of Week 1

If Cell 11 printed `✅ SUCCESS!` and Cell 12 saved the report, the pipeline works.

**Next session (Week 2 preview):**
1. Split the dataset into train/test episodes.
2. Run inference on the test split, compute mean L1 error between predicted and ground-truth actions.
3. That's your **zero-shot baseline number** — the "before" we'll improve on with LoRA fine-tuning in Week 3.

**Don't forget:**
- File → Save copy in Drive
- Push to your GitHub repo (`smolvla-vla-efficiency` or similar)
- Note the VRAM numbers — they're evidence for the final writeup
